# Modulo 2: Supervised Learning e Ottimizzazione
## Predizione del PEGI dei Videogiochi di Steam

### Flusso del progetto:
1. **Feature Engineering** – Caricamento dati, encoding generi, mappatura PEGI
2. **Setup** – Installazione librerie, feature selection, validazione cross-fold
3. **Ottimizzazione XGBoost** – Hyperparameter tuning con Optuna (30 trial)
4. **Ottimizzazione LightGBM** – Hyperparameter tuning con Optuna (20 trial)
5. **Ensemble Stacking** – 4 base learner + meta-learner LogisticRegression
6. **Valutazione Finale** – Metriche, confusion matrix, ROC curve, SHAP values

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MultiLabelBinarizer
import ast
import warnings
warnings.filterwarnings('ignore')

print('[INFO] Sezione 1: Feature Engineering')
print('=' * 70)

[INFO] Sezione 1: Feature Engineering


## Sezione 1: Feature Engineering e Caricamento Dati

In [6]:
# Caricamento dataset (supporto locale e Colab)
csv_path = None
for candidate in ['/content/for_EDA.csv', 'for_EDA.csv']:
    if os.path.exists(candidate):
        csv_path = candidate
        break

if csv_path is None:
    raise FileNotFoundError('[ERROR] for_EDA.csv non trovato!')

df = pd.read_csv(csv_path)
print(f'[INFO] Dataset caricato: {df.shape[0]} righe × {df.shape[1]} colonne')
print(f'[INFO] Colonne: {list(df.columns[:10])}...')


[INFO] Dataset caricato: 103976 righe × 21 colonne
[INFO] Colonne: ['title', 'developer', 'publisher', 'overall_reviews', 'tags', 'genre', 'ram', 'year', 'month', 'user_rating_all']...


In [ ]:
# Ispeziona struttura del dataset
print('[INFO] Struttura del dataset for_EDA.csv:')
print(f'\nColonne disponibili ({len(df.columns)}):')
print(df.columns.tolist())
print(f'\nTipi di dati:')
print(df.dtypes)
print(f'\nPrime righe:')
df.head()

,title,developer,publisher,overall_reviews,tags,genre,ram,year,month,user_rating_all,...,user_rating_recent,total_review_recent,supported_language,price,pegi_rated,age_rating,windows,mac,linux,VR
0,Apex Legends™,['Respawn Entertainment'],['Electronic Arts'],Very Positive,"['Free to Play', 'Battle Royale', 'Multiplayer...","['Action', 'Adventure', 'Free to Play']",6 GB,2020.0,Nov,86.0,...,81.0,15998.0,13.0,0.0,0,16.0,1,0,0,0
1,God of War,['Santa Monica Studio'],['PlayStation PC LLC'],Overwhelmingly Positive,"['Action', 'Adventure', 'Singleplayer', 'Story...","['Action', 'Adventure', 'RPG']",8 GB,2022.0,Jan,97.0,...,96.0,1056.0,18.0,729000.0,0,18.0,1,0,0,0
2,ELDEN RING,['FromSoftware Inc.'],"['FromSoftware Inc.', 'Bandai Namco Entertainm...",Very Positive,"['Souls-like', 'Relaxing', 'Dark Fantasy', 'RP...","['Action', 'RPG']",2 GB,2022.0,Feb,90.0,...,92.0,14027.0,14.0,599000.0,0,16.0,1,0,0,0
3,Grand Theft Auto V,['Rockstar North'],['Rockstar Games'],Very Positive,"['Open World', 'Action', 'Multiplayer', 'Autom...","['Action', 'Adventure']",4 GB,2015.0,Apr,85.0,...,90.0,15021.0,13.0,NaN,0,18.0,1,0,0,0
4,Forza Horizon 5,['Playground Games'],['Xbox Game Studios'],Very Positive,"['Racing', 'Open World', 'Driving', 'Multiplay...","['Action', 'Adventure', 'Racing', 'Simulation'...",8 GB,2021.0,Nov,87.0,...,88.0,3005.0,16.0,699000.0,0,NaN,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103971,The Elder Scrolls® Online,['ZeniMax Online Studios'],['Bethesda Softworks'],Very Positive,"['RPG', 'MMORPG', 'Adventure', 'Exploration', ...","['Action', 'Adventure', 'Massively Multiplayer...",3 GB,2017.0,May,83.0,...,83.0,1225.0,5.0,266000.0,0,18.0,1,1,0,0
103972,Deep Rock Galactic,['Ghost Ship Games'],['Coffee Stain Publishing'],Overwhelmingly Positive,"['Co-op', 'PvE', 'FPS', 'Loot', 'Exploration',...",['Action'],6 GB,2020.0,May,97.0,...,95.0,3319.0,15.0,139999.0,0,NaN,1,0,0,0
103973,Warframe,['Digital Extremes'],['Digital Extremes'],Very Positive,"['Free to Play', 'Action RPG', 'RPG', 'Action'...","['Action', 'Free to Play', 'RPG']",4 GB,2013.0,Mar,90.0,...,83.0,3925.0,14.0,0.0,0,18.0,1,0,0,0
103974,V Rising,['Stunlock Studios'],['Stunlock Studios'],Very Positive,"['Early Access', 'Survival', 'Open World', 'Va...","['Action', 'Adventure', 'Massively Multiplayer...",2 GB,2022.0,May,89.0,...,89.0,23072.0,13.0,108999.0,0,NaN,1,0,0,0


In [9]:
# Multi-Hot Encoding sui generi (se la colonna 'genres' esiste)
if 'genres' in df.columns:
    def clean_genres(g):
        if pd.isna(g) or not isinstance(g, str):
            return []
        g = g.strip()
        if g.startswith('[') and g.endswith(']'):
            try:
                return ast.literal_eval(g)
            except (ValueError, SyntaxError):
                pass
        return [x.strip() for x in g.split(',') if x.strip()]

    df['genres_cleaned'] = df['genres'].apply(clean_genres)
    mlb = MultiLabelBinarizer()
    genres_enc = mlb.fit_transform(df['genres_cleaned'])
    genres_cols = [f"Is_{g.replace(' ','_').replace('-','_').replace('/','_')}" for g in mlb.classes_]
    genres_df = pd.DataFrame(genres_enc, columns=genres_cols, index=df.index)
    df_encoded = pd.concat([df.drop(columns=['genres','genres_cleaned']), genres_df], axis=1)
    
    print(f'[INFO] Generi unici codificati: {len(mlb.classes_)}')
else:
    print('[INFO] Colonna "genres" non trovata, utilizzo il dataframe così com\'è')
    df_encoded = df.copy()

print(f'[INFO] Shape dopo multi-hot encoding: {df_encoded.shape}')

[INFO] Colonna "genres" non trovata, utilizzo il dataframe così com'è
[INFO] Shape dopo multi-hot encoding: (103976, 21)


In [11]:
# Mappatura target PEGI → classi [0-4]
# Cerca colonna target (pegi, required_age, age_rating, etc.)
pegi_col = None
for candidate in ['pegi', 'required_age', 'age_rating', 'rating']:
    if candidate in df_encoded.columns:
        pegi_col = candidate
        break

if pegi_col is None:
    print('[WARNING] Nessuna colonna PEGI trovata!')
    print(f'Colonne disponibili: {df_encoded.columns.tolist()}')
    raise ValueError('Impossibile trovare colonna target per PEGI')

pegi_mapping = {3: 0, 7: 1, 12: 2, 16: 3, 18: 4}

df_encoded['target_class'] = df_encoded[pegi_col].map(pegi_mapping)
df_encoded = df_encoded.dropna(subset=['target_class'])
df_encoded['target_class'] = df_encoded['target_class'].astype(int)
df_final = df_encoded.drop(columns=[pegi_col])

print(f'[INFO] Colonna target utilizzata: "{pegi_col}"')
print('[INFO] Distribuzione classi target:')
for pv, ci in sorted(pegi_mapping.items()):
    count = (df_final['target_class'] == ci).sum()
    pct = count / len(df_final) * 100 if len(df_final) > 0 else 0
    print(f'   PEGI {pv} (Classe {ci}): {count:5d} campioni ({pct:5.1f}%)')

print(f'\n[INFO] Shape dataset finale: {df_final.shape}')

[INFO] Colonna target utilizzata: "age_rating"
[INFO] Distribuzione classi target:
   PEGI 3 (Classe 0):  3954 campioni ( 23.9%)
   PEGI 7 (Classe 1):  1923 campioni ( 11.6%)
   PEGI 12 (Classe 2):  5191 campioni ( 31.4%)
   PEGI 16 (Classe 3):  3164 campioni ( 19.1%)
   PEGI 18 (Classe 4):  2292 campioni ( 13.9%)

[INFO] Shape dataset finale: (16524, 21)


## Sezione 2: Setup – Installazione librerie e Feature Selection

In [12]:
!pip install -q optuna xgboost lightgbm catboost imbalanced-learn shap scikit-learn

"pip" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


In [13]:
import joblib
import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from imblearn.over_sampling import BorderlineSMOTE

optuna.logging.set_verbosity(optuna.logging.WARNING)

print('[INFO] Sezione 2: Setup e configurazione')
print('=' * 70)


ModuleNotFoundError: No module named 'optuna'

In [ ]:
# Suddivisione features e target
X = df_final.drop(columns=['target_class'])
y = df_final['target_class'].values

print(f'[INFO] Shape X: {X.shape} | Shape y: {y.shape}')

# Feature Selection con Mutual Information
K_FEATURES = min(200, X.shape[1])
print(f'[INFO] Selezione top-{K_FEATURES} feature via mutual_info_classif...')

selector = SelectKBest(score_func=mutual_info_classif, k=K_FEATURES)
X_sel_arr = selector.fit_transform(X, y)
sel_names = X.columns[selector.get_support()].tolist()
X_sel = pd.DataFrame(X_sel_arr, columns=sel_names)

print(f'[INFO] Feature selezionate: {X_sel.shape[1]}')
joblib.dump(selector, 'feature_selector.pkl')
joblib.dump(sel_names, 'selected_features.pkl')


In [ ]:
# Configurazione validazione e bilanciamento
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
smt = BorderlineSMOTE(random_state=42, k_neighbors=5, kind='borderline-1')

print(f'[INFO] StratifiedKFold: {N_SPLITS} fold')
print(f'[INFO] Bilanciamento: BorderlineSMOTE (kind=borderline-1)')
print(f'[INFO] Setup completato!\n')


## Sezione 3a: Ottimizzazione XGBoost con Optuna (30 trial)

In [ ]:
print('[INFO] Sezione 3a: Ottimizzazione XGBoost')
print('=' * 70)

def objective_xgb(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': 5,
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': -1,
        # Struttura alberi
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        # Learning
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        # Campionamento stocastico
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        # Regolarizzazione
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }

    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_sel, y):
        Xtr, Xvl = X_sel.iloc[tr_idx], X_sel.iloc[vl_idx]
        ytr, yvl = y[tr_idx], y[vl_idx]
        try:
            Xtr_r, ytr_r = smt.fit_resample(Xtr, ytr)
        except Exception:
            Xtr_r, ytr_r = Xtr, ytr
        
        model = xgb.XGBClassifier(**params)
        model.fit(Xtr_r, ytr_r, verbose=False)
        fold_scores.append(f1_score(yvl, model.predict(Xvl), average='macro'))

    return np.mean(fold_scores)

study_xgb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

print(f'\n[XGBoost] Miglior F1 Macro: {study_xgb.best_value:.4f}')
print('Migliori parametri XGBoost:')
for k, v in study_xgb.best_params.items():
    print(f'   {k}: {v}')


## Sezione 3b: Ottimizzazione LightGBM con Optuna (20 trial)

In [ ]:
print('\n[INFO] Sezione 3b: Ottimizzazione LightGBM')
print('=' * 70)

def objective_lgb(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 5,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }

    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_sel, y):
        Xtr, Xvl = X_sel.iloc[tr_idx], X_sel.iloc[vl_idx]
        ytr, yvl = y[tr_idx], y[vl_idx]
        try:
            Xtr_r, ytr_r = smt.fit_resample(Xtr, ytr)
        except Exception:
            Xtr_r, ytr_r = Xtr, ytr
        
        model = lgb.LGBMClassifier(**params)
        model.fit(Xtr_r, ytr_r)
        fold_scores.append(f1_score(yvl, model.predict(Xvl), average='macro'))

    return np.mean(fold_scores)

study_lgb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)
study_lgb.optimize(objective_lgb, n_trials=20, show_progress_bar=True)

print(f'\n[LightGBM] Miglior F1 Macro: {study_lgb.best_value:.4f}')
print('Migliori parametri LightGBM:')
for k, v in study_lgb.best_params.items():
    print(f'   {k}: {v}')


## Sezione 4: Addestramento Ensemble – StackingClassifier

In [ ]:
print('\n[INFO] Sezione 4: Addestramento Ensemble Stacking')
print('=' * 70)

best_xgb_p = {
    **study_xgb.best_params,
    'objective': 'multi:softprob',
    'num_class': 5,
    'eval_metric': 'mlogloss',
    'random_state': 42,
    'n_jobs': -1
}

best_lgb_p = {
    **study_lgb.best_params,
    'objective': 'multiclass',
    'num_class': 5,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

xgb_final = xgb.XGBClassifier(**best_xgb_p)
lgb_final = lgb.LGBMClassifier(**best_lgb_p)

cat_final = CatBoostClassifier(
    loss_function='MultiClass', random_state=42,
    thread_count=-1, iterations=350, learning_rate=0.05,
    depth=7, l2_leaf_reg=3.0, verbose=0
)

et_final = ExtraTreesClassifier(
    n_estimators=300, random_state=42, n_jobs=-1,
    class_weight='balanced', max_features='sqrt'
)

print('[INFO] Base learner creati:')
print('   - XGBoost (ottimizzato)')
print('   - LightGBM (ottimizzato)')
print('   - CatBoost (iperparametri robusti)')
print('   - ExtraTrees (elevata varianza)')


In [ ]:
# Meta-learner
meta_learner = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, C=1.0, multi_class='multinomial', random_state=42)
)

stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', xgb_final),
        ('lgb', lgb_final),
        ('cat', cat_final),
        ('et', et_final),
    ],
    final_estimator=meta_learner,
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)

print('[INFO] StackingClassifier configurato (4 base + LogReg meta-learner)')

# Bilanciamento
print('[INFO] Applicando BorderlineSMOTE...')
smt_final = BorderlineSMOTE(random_state=42, k_neighbors=5, kind='borderline-1')
X_res, y_res = smt_final.fit_resample(X_sel, y)
print(f'[INFO] Bilanciamento: {X_sel.shape} → {X_res.shape}')

# Addestramento
print('[INFO] Avvio addestramento StackingClassifier...')
stacking_clf.fit(X_res, y_res)
print('[INFO] Addestramento completato!')

# Salvataggio
joblib.dump(stacking_clf, 'stacking_pegi_model.pkl')
joblib.dump(sel_names, 'model_features.pkl')
print('[INFO] Modelli salvati con successo!')


## Sezione 5: Valutazione Finale e Analisi

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score, roc_auc_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import shap

print('\n[INFO] Sezione 5: Valutazione Finale')
print('=' * 70)

y_pred = stacking_clf.predict(X_sel)
y_proba = stacking_clf.predict_proba(X_sel)
pegi_labels = [f'PEGI {k}' for k in sorted(pegi_mapping.keys())]

print('\n1. CLASSIFICATION REPORT\n')
print('=' * 70)
print(classification_report(y, y_pred, target_names=pegi_labels))
print('=' * 70)


In [ ]:
# Metriche avanzate
kappa = cohen_kappa_score(y, y_pred)
mcc = matthews_corrcoef(y, y_pred)
bal_acc = balanced_accuracy_score(y, y_pred)
roc_auc = roc_auc_score(y, y_proba, multi_class='ovr', average='macro')

print('\n2. METRICHE AVANZATE\n')
print(f'  Cohen Kappa:          {kappa:.4f}')
print(f'  Matthews Corr. Coef: {mcc:.4f}')
print(f'  Balanced Accuracy:   {bal_acc:.4f}')
print(f'  Macro ROC-AUC:       {roc_auc:.4f}')


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

print('\n3. MATRICE DI CONFUSIONE\n')
plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=pegi_labels, yticklabels=pegi_labels,
            cbar_kws={'label': 'Frequenza (%)'})
plt.title('Matrice di Confusione Normalizzata (%)', fontsize=14, fontweight='bold')
plt.ylabel('Classe Reale', fontsize=12)
plt.xlabel('Classe Predetta', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ROC Curves
y_bin = label_binarize(y, classes=[0, 1, 2, 3, 4])
colors_roc = ['steelblue', 'darkorange', 'green', 'crimson', 'purple']

print('\n4. CURVE ROC MULTI-CLASSE\n')
plt.figure(figsize=(10, 7))
for i, (lab, col) in enumerate(zip(pegi_labels, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_i = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=col, lw=2, label=f'{lab} (AUC = {roc_i:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Multi-Classe', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance
imp_scores = stacking_clf.named_estimators_['xgb'].feature_importances_
imp_df = pd.DataFrame({'Feature': sel_names, 'Importance': imp_scores})
imp_df = imp_df.sort_values('Importance', ascending=False)

print('\n5. TOP-25 FEATURE IMPORTANCE\n')
top_n = min(25, len(imp_df))
plt.figure(figsize=(13, 9))
sns.barplot(data=imp_df.head(top_n), x='Importance', y='Feature',
            palette='viridis', hue='Feature', legend=False)
plt.title(f'Top-{top_n} Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# SHAP Analysis
print('\n6. SHAP VALUE ANALYSIS\n')
explainer = shap.TreeExplainer(stacking_clf.named_estimators_['xgb'])
sample_size = min(300, len(X_sel))
X_sample = X_sel.sample(n=sample_size, random_state=42)
shap_values = explainer.shap_values(X_sample)

plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type='bar',
                  class_names=pegi_labels, show=False)
plt.title('SHAP Feature Importance per Classe', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [1]:
# Feature Group Analysis
groups = {
    'Spatial Grid': [c for c in sel_names if c.startswith('grid_')],
    'Momenti LAB': [c for c in sel_names if any(c.startswith(p) for p in ['L_', 'a_', 'b_'])],
    'LBP Texture': [c for c in sel_names if c.startswith('lbp_')],
    'GLCM': [c for c in sel_names if c.startswith('glcm_')],
    'HOG': [c for c in sel_names if c.startswith('hog_')],
    'HSV': [c for c in sel_names if c.startswith('hsv_')],
    'Edge/Sharp': [c for c in sel_names if c in ['sharpness_laplacian', 'edge_density_canny']],
    'TF-IDF': [c for c in sel_names if c.startswith('tfidf_')],
    'Generi': [c for c in sel_names if c.startswith('Is_')],
    'Prezzo': [c for c in sel_names if c == 'price'],
}

total_imp = imp_df['Importance'].sum()
print('\n7. ANALISI CONTRIBUTO PER GRUPPO\n')
print('=' * 80)
for grp_name, feat_list in groups.items():
    grp_imp = imp_df[imp_df['Feature'].isin(feat_list)]['Importance'].sum()
    pct = (grp_imp / total_imp * 100) if total_imp > 0 else 0.0
    n_sel = len([f for f in feat_list if f in sel_names])
    print(f'  {grp_name:<20} imp={grp_imp:.4f} ({pct:5.1f}%)  [{n_sel} feat]')
print('=' * 80)
print('\n[INFO] Notebook completato con successo!')


NameError: name 'sel_names' is not defined